In [2]:
!pip install requests
!pip install aria2
import subprocess

# Get download URL first
import requests
record_id = "17686067"
api_url = f"https://zenodo.org/api/records/{record_id}"
record = requests.get(api_url).json()

# Download with aria2c (16 parallel connections)
for file_info in record['files']:
    if file_info['key'].endswith('.pkl'):
        url = file_info['links']['self']
        filename = file_info['key']

        print(f"Downloading {filename} with aria2c...")
        subprocess.run([
            'aria2c',
            '-x16',  # 16 connections
            '-s16',  # 16 splits
            '--file-allocation=none',
            f'--out={filename}',
            url
        ])
        break


In [3]:

import numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
import random
with open("/content/processed_mosei.pkl", "rb") as f:
  data = pickle.load(f)
for key in data:
  data_point = data[key]
  for x in data_point:
    x["Labels"]["features"] = torch.tensor(x["Labels"]["features"][1:])
random.seed(42)
key_list = list(data.keys())
random.shuffle(key_list)
train_keys = key_list[:int(0.7*len(key_list))]
valiation_keys = key_list[int(0.7*len(key_list)):int(0.85*len(key_list))]
test_keys = key_list[int(0.85*len(key_list)):]
train_data = []
val_data = []
test_data = []

# Populate train_data
for vid in train_keys:
    for utterance in data[vid]:
        train_data.append((vid, utterance))

# Populate val_data
for vid in valiation_keys:
    for utterance in data[vid]:
        val_data.append((vid, utterance))

# Populate test_data
for vid in test_keys:
    for utterance in data[vid]:
        test_data.append((vid, utterance))


In [4]:
class TextDataset(Dataset):
    """
    Utterance-level CMU-MOSEI dataset.
    items: list of (video_id, utt_dict)
    """
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        vid, utt = self.items[idx]
        # --- text ---
        t_feat = utt["WORDVEC"]["features"]
        if t_feat.shape[0] == 0:  # If visual feature is empty
            t_feat = np.zeros((1, 300), dtype=np.float32) # Replace with a single zero-padded frame
        t_feat = np.nan_to_num(t_feat, nan=0.0, posinf=0.0, neginf=0.0)
        t_feat = torch.from_numpy(t_feat).float()

        # --- label ---
        y = utt["Labels"]["features"]
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        y = torch.from_numpy(y)

        return t_feat, y

def pad_collate_fn(batch):
    """
    batch: list of tuples (seq, label)
        seq   = Tensor (T, E)
        label = Tensor (num_classes,)
    """
    sequences = [x[0] for x in batch]
    labels = [x[1] for x in batch]

    # 1. Find max length in this batch
    max_len = max(seq.size(0) for seq in sequences)

    padded_seqs = []
    for seq in sequences:
        T, E = seq.size()
        pad_len = max_len - T

        if pad_len > 0:
            # pad with zeros → shape = (pad_len, E)
            pad_tensor = torch.zeros(pad_len, E)
            seq = torch.cat([seq, pad_tensor], dim=0)

        padded_seqs.append(seq)

    # 2. Stack into batch tensors
    X = torch.stack(padded_seqs, dim=0)  # (B, T, E)
    Y = torch.stack(labels, dim=0)       # (B, num_classes)

    return X, Y

batch_size = 10

train_dataset = TextDataset(train_data)
val_dataset   = TextDataset(val_data)
test_dataset  = TextDataset(test_data)



In [6]:
c = 0
for x, i in test_dataset:
  print(i)
  c += 1
  if c == 10:
    break

tensor([0.0000, 2.0000, 0.0000, 0.0000, 0.0000, 1.6667])
tensor([0.6667, 0.3333, 0.3333, 0.3333, 0.3333, 0.6667])
tensor([0.0000, 1.6667, 0.0000, 0.0000, 0.0000, 0.3333])
tensor([0.6667, 0.0000, 0.3333, 0.0000, 0.0000, 0.0000])
tensor([0.0000, 0.3333, 0.3333, 0.0000, 0.3333, 0.6667])
tensor([0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])
tensor([0., 0., 0., 0., 0., 0.])
tensor([0.0000, 0.0000, 0.3333, 0.3333, 0.0000, 0.0000])
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.6667, 0.0000])
tensor([0.0000, 0.6667, 0.0000, 0.0000, 0.6667, 0.0000])


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Attention(nn.Module):
    def __init__(self, hidden_dim, dropout=0.15):
        super().__init__()
        self.attn_w = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, H):
        # H: (B, T, hidden_dim)
        scores = self.attn_w(self.dropout(H)).squeeze(-1)  # (B, T)
        weights = torch.softmax(scores, dim=1)             # (B, T)
        context = torch.sum(weights.unsqueeze(-1) * H, dim=1)  # (B, hidden_dim)
        return context, weights


class EmotionCNNBiLSTM(nn.Module):
    def __init__(self, embed_dim, lstm_hidden=128,
                 num_classes=6, num_filters=128,
                 kernel_sizes=[3,4,5], dropout=0.15):

        super().__init__()

        self.dropout = nn.Dropout(dropout)

        # BiLSTM input_dim = embed_dim from your dataset (300)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=True
        )
        lstm_outdim = lstm_hidden * 2

        # Attention layer on top of BiLSTM
        self.attention = Attention(lstm_outdim, dropout=dropout)

        # CNN input = concat(LSTM_H, context_expanded)
        cnn_input_dim = lstm_outdim * 2

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=cnn_input_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        # Fully connected layers
        self.fc1 = nn.Linear(len(kernel_sizes) * num_filters, 100)
        self.fc2 = nn.Linear(100, num_classes)
        self.final_dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, T, E) embeddings directly
        B, T, _ = x.size()

        # Apply dropout to embeddings
        emb = self.dropout(x)

        # BiLSTM → H shape = (B, T, 2H)
        H, _ = self.lstm(emb)

        # Attention → context: (B, 2H)
        context, attn_weights = self.attention(H)

        # Expand context over time
        context_expanded = context.unsqueeze(1).expand(-1, T, -1)

        # CNN input: concat along feature dimension
        cnn_input = torch.cat([H, context_expanded], dim=-1)  # (B, T, 4H)

        # CNN expects (B, C_in, T)
        cnn_input = cnn_input.transpose(1, 2)

        # Apply multiple convolution kernels
        conv_results = [F.relu(conv(cnn_input)) for conv in self.convs]
        pooled = [F.max_pool1d(c, c.size(2)).squeeze(2) for c in conv_results]

        # Combine all filter outputs
        cnn_cat = torch.cat(pooled, dim=1)

        # Classifier
        x = F.relu(self.fc1(cnn_cat))
        x = self.final_dropout(x)
        logits = self.fc2(x)

        return logits, attn_weights


In [23]:
def train_one_epoch(model, dataloader, optimizer, criterion, device, metric):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for inputs, labels in dataloader:

        # inputs: (B, T)
        # labels: (B,)

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits, _ = model(inputs)    # model respects sequence order

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)

        total_samples += labels.size(0)
        metric.update(labels, logits)

    avg_loss = total_loss / total_samples
    return avg_loss
def eval(model, dataloader, device, criterion, metric):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            logits, _ = model(inputs)
            loss = criterion(logits, labels)
            total_loss += loss.item() * inputs.size(0)
            total_samples += labels.size(0)
            metric.update(labels, logits)
    avg_loss = total_loss / total_samples
    v = metric.compute()
    return v[0], v[1], avg_loss

In [24]:
import torch
import numpy as np
from sklearn.metrics import f1_score

# ---------- Weighted Accuracy ----------
def weighted_accuracy(y_true, y_pred):
    """
    y_true, y_pred: numpy arrays of shape (N, 6) or (N,)
    """
    diff = np.abs(y_true - y_pred)
    weights = 1 - diff / 3
    return np.clip(weights, 0, 1).mean()


# ---------- F1 for emotion presence ----------
def f1_binary_emotion(y_true, y_pred):
    """
    Convert intensities 0-3 to binary (present vs absent)
    """
    y_true_bin = (y_true > 0).astype(int)
    y_pred_bin = (y_pred > 0).astype(int)

    return f1_score(y_true_bin, y_pred_bin, average='weighted')


# ---------- EPISODE / BATCH ACCUMULATION ----------
class MetricTracker:
    def __init__(self):
        self.y_true_all = []
        self.y_pred_all = []

    def update(self, y_true_batch, y_pred_batch):
        """
        y_true_batch: torch tensor (B, 6)
        y_pred_batch: torch tensor (B, 6) logits or raw predictions
        """
        # Convert to CPU numpy safely
        y_true_np = y_true_batch.detach().cpu().numpy()
        y_pred_np = y_pred_batch.detach().cpu().numpy()

        # If model outputs logits/probs, convert to intensity predictions
        # Example: assume model outputs 4-class logits for each emotion (B, 6, 4)
        if y_pred_np.ndim == 3:
            y_pred_np = np.argmax(y_pred_np, axis=2)  # take intensity 0,1,2,3

        self.y_true_all.append(y_true_np)
        self.y_pred_all.append(y_pred_np)

    def compute(self):
        """
        Computes WA and F1 over all batches.
        """
        y_true = np.concatenate(self.y_true_all, axis=0)
        y_pred = np.concatenate(self.y_pred_all, axis=0)

        wa = weighted_accuracy(y_true, y_pred)
        f1 = f1_binary_emotion(y_true, y_pred)

        return wa, f1


In [25]:
import random
import warnings
warnings.filterwarnings('ignore')

search_space = {
    "lstm_hidden": [64, 128, 256],
    "num_filters": [64, 128, 256],
    "kernel_sizes": [[3,4,5], [2,3,4], [3,5,7]],
    "dropout": [0.1, 0.2, 0.3, 0.5],
    "lr": [1e-4, 3e-4, 1e-3],
    "batch_size": [8, 16, 32]
}
best_val_f1 = 0.0
best_model = None
best_cfg = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")
def sample_config():
    return {
        "lstm_hidden": random.choice(search_space["lstm_hidden"]),
        "num_filters": random.choice(search_space["num_filters"]),
        "kernel_sizes": random.choice(search_space["kernel_sizes"]),
        "dropout": random.choice(search_space["dropout"]),
        "lr": random.choice(search_space["lr"]),
        "batch_size": random.choice(search_space["batch_size"]),
    }
for trial in range(10):   # try 10 configs
    cfg = sample_config()
    batch_size = cfg["batch_size"]
    print("Trial", trial, cfg)
    model = EmotionCNNBiLSTM(
        embed_dim=300,
        lstm_hidden=cfg["lstm_hidden"],
        num_filters=cfg["num_filters"],
        kernel_sizes=cfg["kernel_sizes"],
        dropout=cfg["dropout"],
        num_classes=6
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(
      train_dataset,
      batch_size=batch_size,
      shuffle=True,
      collate_fn=pad_collate_fn,
    )

    val_loader = DataLoader(
      val_dataset,
      batch_size=batch_size,
      shuffle=False,
      collate_fn=pad_collate_fn,
    )

    test_loader = DataLoader(
      test_dataset,
      batch_size=batch_size,
      shuffle=False,
      collate_fn=pad_collate_fn,
    )
    for epoch in range(5):   # fast evaluation
        train_metric = MetricTracker()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, train_metric)
        train_wa, train_f1 = train_metric.compute()
        val_metric = MetricTracker()
        val_wa, val_f1, val_loss = eval(model, val_loader, device, criterion, val_metric)

        print(f"Epoch {epoch}, Train loss: {train_loss}, Train Weight average: {train_wa}, Train f1: {train_f1}")
        print(f"Epoch {epoch}, Val loss: {val_loss}, Val Weight average: {val_wa}, Val f1: {val_f1}")
        if val_f1 > best_val_f1:
            best_cfg = cfg
            best_model = model
            best_val_f1 = val_f1
    print(f"Best validation accuracy: {best_val_f1}")

Using cuda
Trial 0 {'lstm_hidden': 128, 'num_filters': 128, 'kernel_sizes': [3, 5, 7], 'dropout': 0.3, 'lr': 0.0001, 'batch_size': 16}
Epoch 0, Train loss: 1.53296866238932, Train Weight average: 0.8366665840148926, Train f1: 0.3951934178651494
Epoch 0, Val loss: 1.582913211740244, Val Weight average: 0.7852430939674377, Val f1: 0.41719822699738923
Epoch 1, Train loss: 1.4568767667944875, Train Weight average: 0.7967100143432617, Train f1: 0.4396270632123384
Epoch 1, Val loss: 1.5168056041668778, Val Weight average: 0.8178473711013794, Val f1: 0.4558641961951736
Epoch 2, Train loss: 1.4360439634473543, Train Weight average: 0.7922597527503967, Train f1: 0.4491053703088815
Epoch 2, Val loss: 1.5045793675766965, Val Weight average: 0.7899308204650879, Val f1: 0.45701253703694455
Epoch 3, Train loss: 1.4248510487573505, Train Weight average: 0.785812258720398, Train f1: 0.45671827715713104
Epoch 3, Val loss: 1.4858916914120268, Val Weight average: 0.8030292391777039, Val f1: 0.46795737633

Eval the accuracy for test set

In [27]:
test_metric = MetricTracker()
criterion = nn.CrossEntropyLoss()
test_wa, test_f1, test_loss = eval(best_model, test_loader, device, criterion, test_metric)
print(f"Test loss: {test_loss}, Test Weight average: {test_wa}, Test f1: {test_f1}")

Test loss: 1.452822626870254, Test Weight average: 0.7821335792541504, Test f1: 0.4910495246799324
